# sre-gym Basic-tier training — Unsloth + TRL GRPO

> **End-to-end training pipeline for a 3B specialist on a single A100, ~12h, ~$30 of HF credits.**

## What this notebook does

1. Loads Qwen2.5-3B-Instruct in 4-bit via Unsloth (fallback path: Qwen2.5-1.5B for L4 / 24GB GPUs).
2. Boots the sre-gym Basic-tier env locally inside the Colab VM.
3. Builds a seed SFT dataset by replaying the scripted-optimal baselines across all 12 Basic templates × 5 procgen variants (60 train / 12 eval). Each step becomes a (prompt, response) pair.
4. SFT cold-starts the Qwen LoRA on the seed corpus (~3h on A100).
5. Runs GRPO online — for each step, samples a scenario, rolls out K=4 trajectories, computes group-relative advantages from the env's deterministic grader, applies a policy gradient update (~6h on A100).
6. Evaluates the trained adapter against the held-out 12 procgen variants and writes the comparison row.
7. Pushes the trained adapter to HuggingFace Hub.

## Tier framing reminder

This is the **Basic** tier — bounded by **compute**. Pre-digested observations (~600 tokens), 11 actions, 12-tick episodes, 5-component dense rubric reward. Everything is tuned to fit a 3B GRPO run inside a 12h A100 budget.

The Advanced and Max tiers are designed but not trained in this repo — see [`docs/ADVANCED_TIER.md`](../docs/ADVANCED_TIER.md) and [`docs/MAX_TIER.md`](../docs/MAX_TIER.md).

## Hardware + cost

| GPU | Wall-clock | Compute cost |
|---|---|---|
| A100 40GB (HF Pro Spaces or Colab A100) | ~12h end-to-end | ~$30 of HF credits |
| L4 24GB (HF Pro Free) | ~22h, drop to Qwen2.5-1.5B | ~$0 (free tier) |
| H100 80GB | ~6h, can use Qwen2.5-7B | ~$60 of HF credits |

## 1. Install dependencies

Unsloth pinned because `>=2025.1.0` ships the Qwen2.5 fast-path. TRL pinned to `>=0.12.0` for the modern GRPO trainer.

In [ ]:
%%bash
pip install -q --upgrade pip
pip install -q 'unsloth>=2025.1.0' 'unsloth_zoo>=2025.1.0'
pip install -q 'trl>=0.12.0' 'transformers>=4.46' 'peft>=0.13' 'accelerate>=1.1' 'bitsandbytes>=0.44'
pip install -q 'datasets>=3.0' 'wandb>=0.18'
pip install -q 'openenv-core>=0.2.1' 'fastapi>=0.115' 'uvicorn[standard]>=0.30' 'pydantic>=2.8'
pip install -q 'httpx>=0.27' 'matplotlib' 'pandas' 'pyyaml'

In [ ]:
import os
import sys
import json
import random
import time
from pathlib import Path
import subprocess

# Adjust this if you cloned the repo somewhere else.
REPO_ROOT = Path('/content/sre-enginnerllm')
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/dakshdoesdev/sre-enginnerllm.git', str(REPO_ROOT)])
sys.path.insert(0, str(REPO_ROOT))

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device         : {torch.cuda.get_device_name(0)}')
    print(f'Memory (GB)    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}')

GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if GPU_VRAM_GB >= 38:
    BASE_MODEL = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
    LORA_RANK = 64
    GRPO_K = 4
elif GPU_VRAM_GB >= 22:
    BASE_MODEL = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
    LORA_RANK = 32
    GRPO_K = 2
else:
    BASE_MODEL = 'unsloth/Qwen2.5-0.5B-Instruct'
    LORA_RANK = 16
    GRPO_K = 2
print(f'Selected base model: {BASE_MODEL} (LoRA r={LORA_RANK}, GRPO K={GRPO_K})')

## 2. Boot the sre-gym env locally

We start the FastAPI server in the background and let it serve `/reset`, `/step`, `/state`, `/tasks`, `/baseline`, and `/grader`. The training loop talks to it over HTTP.

In [ ]:
import httpx
import threading
import uvicorn

from unified_incident_env.server.app import create_compatible_app
from unified_incident_env.server.challenge import list_scenarios, list_baselines, get_scenario

ENV_PORT = 8001
ENV_BASE = f'http://127.0.0.1:{ENV_PORT}'

_server_started = False
def _run_server():
    app = create_compatible_app()
    uvicorn.run(app, host='127.0.0.1', port=ENV_PORT, log_level='warning')

if not _server_started:
    threading.Thread(target=_run_server, daemon=True).start()
    _server_started = True
    time.sleep(3)
    print(f'Env booted at {ENV_BASE}')

with httpx.Client(timeout=10.0) as client:
    health = client.get(f'{ENV_BASE}/health').json()
    catalog = client.get(f'{ENV_BASE}/tasks').json()
print('health  :', health)
print('scenarios:', len(catalog['scenarios']))
templates = sorted({s['id'].split('__')[0] for s in catalog['scenarios']})
print(f'{len(templates)} templates: {templates}')

## 3. Held-out / training scenario split

60 train / 12 eval. Holdout = the `__p05` slice (one variant per template).


In [ ]:
ALL_SCENARIO_IDS = [s['id'] for s in catalog['scenarios']]
HOLDOUT_IDS  = [s['id'] for s in catalog['scenarios'] if s['id'].endswith('__p05')]
TRAIN_IDS    = [s['id'] for s in catalog['scenarios'] if s['id'] not in HOLDOUT_IDS]
print(f'Train : {len(TRAIN_IDS)} scenarios')
print(f'Holdout: {len(HOLDOUT_IDS)} scenarios')
for h in HOLDOUT_IDS:
    print(f'  {h}')
Path('/content/sre-train-data').mkdir(exist_ok=True, parents=True)

## 4. Build SFT seed dataset by replaying scripted-optimal baselines

Every step in every baseline becomes a (prompt, completion) pair. The completion is the JSON-formatted action with rationale. This is our SFT cold-start corpus.

Total samples = 60 scenarios × ~9 actions/scenario ≈ 540 SFT samples. Plus optional Claude-teacher trajectories from `train/data/seed_combined.jsonl` if present.

In [ ]:
from unified_incident_env.server.environment import UnifiedIncidentEnvironment
from unified_incident_env.models import UnifiedIncidentAction

SYSTEM_PROMPT = '''You are a senior SRE on-call. The environment is a partially-observable
incident with 4 services (api-gateway, cache, database, worker). Use the 11 available
actions to diagnose, remediate, verify, and resolve.

Output a SINGLE JSON object on each turn:
{"action_type": "...", "service": "...", "metric": "...", "check_name": "...", "hypothesis": {...}}
Only include fields the action requires.'''

def render_observation(obs_dict):
    return obs_dict['prompt_text']

def action_to_json(action):
    payload = action.model_dump(exclude_none=True)
    if 'hypothesis' in payload and payload['hypothesis'] is None:
        payload.pop('hypothesis')
    return json.dumps(payload, separators=(',', ':'))

sft_pairs = []
for scenario_id in TRAIN_IDS:
    template_id = get_scenario(scenario_id).get('template_id', scenario_id)
    baseline = list_baselines(scenario_id=scenario_id).baselines[0]
    env = UnifiedIncidentEnvironment()
    obs = env.reset(scenario_id=scenario_id)
    for step in baseline.actions:
        prompt = SYSTEM_PROMPT + '\n\n' + render_observation(obs.model_dump())
        completion = action_to_json(step.action)
        sft_pairs.append({'prompt': prompt, 'completion': completion, 'rationale': step.rationale,
                          'scenario_id': scenario_id, 'template_id': template_id})
        obs = env.step(step.action)
        if obs.done:
            break

print(f'Built {len(sft_pairs)} SFT pairs from {len(TRAIN_IDS)} scenarios')

# Optionally fold in seed_combined.jsonl Claude-teacher data if present.
claude_seed = REPO_ROOT / 'train' / 'data' / 'seed_combined.jsonl'
if claude_seed.exists():
    with claude_seed.open() as f:
        for line in f:
            row = json.loads(line)
            sft_pairs.append({'prompt': row['prompt'], 'completion': row['completion'],
                              'rationale': row.get('rationale', ''),
                              'scenario_id': row.get('scenario_id', ''),
                              'template_id': row.get('template_id', ''),})
    print(f'Folded in {claude_seed} -> total {len(sft_pairs)} SFT pairs')

# Save the SFT dataset for reproducibility.
out = Path('/content/sre-train-data/sft_seed.jsonl')
with out.open('w') as f:
    for row in sft_pairs:
        f.write(json.dumps(row) + '\n')
print(f'Saved -> {out}')

## 5. Load Qwen2.5 in 4-bit via Unsloth, attach a LoRA

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    lora_dropout=0.0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print(f'Model loaded: {BASE_MODEL}')
print(f'Trainable params : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'Total params     : {sum(p.numel() for p in model.parameters()):,}')

## 6. SFT cold start

500 steps, batch 4 × grad-accum 2 = effective batch 8. Cosine LR schedule. Saves every 250 steps.

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

def to_chatml(row):
    text = (
        '<|im_start|>system\n' + SYSTEM_PROMPT + '<|im_end|>\n'
        '<|im_start|>user\n' + row['prompt'] + '<|im_end|>\n'
        '<|im_start|>assistant\n' + row['completion'] + '<|im_end|>'
    )
    return {'text': text}

sft_ds = Dataset.from_list(sft_pairs).map(to_chatml, remove_columns=['prompt', 'completion'])
sft_ds = sft_ds.shuffle(seed=42)
print(f'SFT dataset: {len(sft_ds)} rows')

sft_config = SFTConfig(
    output_dir='/content/sft-out',
    num_train_epochs=2,
    max_steps=500,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=25,
    save_steps=250,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    optim='adamw_8bit',
    report_to='none',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    dataset_text_field='text',
)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=sft_ds, args=sft_config)
sft_stats = trainer.train()
print(f'SFT done. Final loss: {sft_stats.metrics.get("train_loss", "?")}')
model.save_pretrained('/content/sft-out/lora')
tokenizer.save_pretrained('/content/sft-out/lora')

## 7. GRPO online training against the live env

For each step:

1. Sample a scenario from `TRAIN_IDS`.
2. Roll out K trajectories with the current policy (sampling temperature 0.7, max 12 ticks).
3. Compute reward for each trajectory = `final_score` from the env's deterministic grader.
4. Compute group-relative advantages = `(reward_i - mean(rewards)) / std(rewards)`.
5. Apply policy gradient update.

We use TRL's GRPOTrainer with a custom reward function that boots a fresh env, replays the model's output through it, and reads back the final score. This is the OpenClaw-style reward path.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

def make_grpo_prompt(scenario_id):
    env = UnifiedIncidentEnvironment()
    obs = env.reset(scenario_id=scenario_id)
    initial_prompt = (
        '<|im_start|>system\n' + SYSTEM_PROMPT + '<|im_end|>\n'
        '<|im_start|>user\n' + obs.prompt_text + '<|im_end|>\n'
        '<|im_start|>assistant\n'
    )
    return initial_prompt, env, obs

def parse_action(text):
    # Extract first JSON object the model emits; tolerate prose around it.
    text = text.strip()
    if not text.startswith('{'):
        i = text.find('{')
        if i == -1:
            return None
        text = text[i:]
    depth = 0; end = None
    for j, ch in enumerate(text):
        if ch == '{': depth += 1
        elif ch == '}': depth -= 1
        if depth == 0:
            end = j; break
    if end is None:
        return None
    try:
        return json.loads(text[:end+1])
    except Exception:
        return None

def episode_reward(prompts, completions, **kwargs):
    """GRPO reward function — runs the completion against the env.

    The TRL GRPOTrainer calls this with a list of completions per prompt; we
    return one scalar reward per completion. We reset a fresh env per
    completion and step it through with the model's output as the FIRST action;
    everything after is greedy-rolled-out by stepping through subsequent
    samples we'd need a second call for. For Basic-tier scope, single-shot
    reward against the first action correlates well with full-episode reward
    because the per-tick shaping reward is dense.
    """
    rewards = []
    for prompt, completion in zip(prompts, completions):
        scenario_id = kwargs.get('scenario_id', [random.choice(TRAIN_IDS)] * len(prompts))[0] if isinstance(kwargs.get('scenario_id'), list) else kwargs.get('scenario_id', random.choice(TRAIN_IDS))
        env = UnifiedIncidentEnvironment()
        env.reset(scenario_id=scenario_id)
        action_dict = parse_action(completion)
        if action_dict is None:
            rewards.append(-0.05); continue
        try:
            action = UnifiedIncidentAction(**action_dict)
        except Exception:
            rewards.append(-0.05); continue
        obs = env.step(action)
        # First-action shaped reward: combination of env reward + cumulative trajectory
        # Doing 12 forward passes per rollout per K is too expensive at the Basic budget,
        # so we score on the first action's shaped reward as a proxy. The dense shaping
        # ensures this correlates ~0.85 with final episode reward (verified empirically).
        rewards.append(float(obs.reward))
    return rewards

GRPO_TRAIN_PROMPTS = [{'prompt': make_grpo_prompt(sid)[0], 'scenario_id': sid} for sid in TRAIN_IDS]
grpo_ds = Dataset.from_list(GRPO_TRAIN_PROMPTS)

grpo_config = GRPOConfig(
    output_dir='/content/grpo-out',
    num_train_epochs=1,
    max_steps=800,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    optim='adamw_8bit',
    report_to='none',
    max_prompt_length=2048,
    max_completion_length=256,
    num_generations=GRPO_K,
    temperature=0.7,
    beta=0.04,
)

grpo_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[episode_reward],
    args=grpo_config,
    train_dataset=grpo_ds,
)
grpo_stats = grpo_trainer.train()
print(f'GRPO done. Final reward: {grpo_stats.metrics}')
model.save_pretrained('/content/grpo-out/lora')
tokenizer.save_pretrained('/content/grpo-out/lora')

## 8. Evaluate the trained adapter against the 12-scenario held-out set

3 episodes per held-out scenario × 12 scenarios = 36 trajectories. We log per-template reward distributions and write `eval_results.json`.

In [ ]:
FastLanguageModel.for_inference(model)

def run_episode(scenario_id, max_ticks=12, temperature=0.0):
    env = UnifiedIncidentEnvironment()
    obs = env.reset(scenario_id=scenario_id)
    history_messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': obs.prompt_text},
    ]
    for _ in range(max_ticks):
        text = tokenizer.apply_chat_template(history_messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=128, do_sample=temperature > 0,
                             temperature=max(0.05, temperature), pad_token_id=tokenizer.eos_token_id)
        completion = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        action_dict = parse_action(completion)
        if action_dict is None:
            history_messages.append({'role': 'assistant', 'content': completion})
            history_messages.append({'role': 'user', 'content': 'Output a valid JSON action object.'})
            continue
        try:
            action = UnifiedIncidentAction(**action_dict)
        except Exception:
            history_messages.append({'role': 'assistant', 'content': completion})
            history_messages.append({'role': 'user', 'content': 'Output a valid JSON action object.'})
            continue
        obs = env.step(action)
        history_messages.append({'role': 'assistant', 'content': completion})
        history_messages.append({'role': 'user', 'content': obs.prompt_text})
        if obs.done:
            break
    return {
        'scenario_id': scenario_id,
        'final_score': float(obs.final_score),
        'incident_resolved': bool(obs.incident_resolved),
        'tick_count': int(obs.tick_count),
        'breakdown': dict(obs.score_breakdown),
    }

EPISODES_PER_SCENARIO = 3
results = []
for sid in HOLDOUT_IDS:
    for ep in range(EPISODES_PER_SCENARIO):
        r = run_episode(sid, temperature=0.0 if ep == 0 else 0.5)
        r['episode'] = ep
        results.append(r)
        print(f'  {sid:<32} ep{ep} score={r["final_score"]:.3f} resolved={r["incident_resolved"]} ticks={r["tick_count"]}')

Path('/content/eval-results').mkdir(exist_ok=True, parents=True)
with open('/content/eval-results/trained_qwen.jsonl', 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')
print(f'\nTrained Qwen mean score: {sum(r["final_score"] for r in results)/len(results):.3f}')
print(f'Resolved rate: {sum(r["incident_resolved"] for r in results)}/{len(results)}')

## 9. Push the trained adapter to HuggingFace Hub

Set `HF_TOKEN` in Colab Secrets. The merged-adapter checkpoint is small (~50MB at LoRA r=64) so this is fast.

In [ ]:
import os
from google.colab import userdata    # Colab-only convenience
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    print('HF_TOKEN not set in Colab Secrets — skipping push step.')

REPO_ID = 'dakshdoesdev/sre-gym-qwen25-3b-grpo'
if 'HF_TOKEN' in os.environ:
    model.push_to_hub(REPO_ID, token=os.environ['HF_TOKEN'])
    tokenizer.push_to_hub(REPO_ID, token=os.environ['HF_TOKEN'])
    print(f'Pushed to {REPO_ID}')
else:
    print('Skipped push.')

## 10. Next steps

- **Comparison plots**: see [`02_basic_eval_comparison.ipynb`](02_basic_eval_comparison.ipynb) — runs random / heuristic / Llama-3.3-70B / Claude Haiku / Claude Sonnet against the same held-out set and produces the comparison table.
- **Advanced tier walkthrough**: [`03_advanced_blueprint_walkthrough.ipynb`](03_advanced_blueprint_walkthrough.ipynb) — manual play-through of `cascading_release_train.yaml` to demonstrate the chained-incident reasoning gap.
- **Max tier reference trace**: [`04_max_demo_chaos.ipynb`](04_max_demo_chaos.ipynb) — what a 110-action trajectory against the e-commerce family looks like.

## Why this loop is correct for the Basic tier

1. **Compute-bounded by design**: ~12h on A100. Larger models / longer rollouts violate the Basic-tier persona ("$30 of HF credits").
2. **Dense shaping reward**: the per-tick `obs.reward` is shaped (potential-function differences) so single-action GRPO scoring correlates ~0.85 with full-episode reward. See [`docs/REWARD_DESIGN.md`](../docs/REWARD_DESIGN.md) §3.
3. **Group-relative advantages**: GRPO's K=4 rollouts per scenario keep training stable without a learned-critic — appropriate for the deterministic env.
4. **Held-out evaluation**: the `__p05` slice is never seen during training, so reported numbers are honest.